## Importação de Bibliotecas

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Avisos
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

### Parâmetros globais

In [ ]:
# define a coluna alvo do modelo
TARGET = 'FPD'

## Carregamento dos Dados

In [ ]:
# Carregar dados
abt01_train = pd.read_parquet(PROCESSED_DIR / 'abt01_train_fs.parquet')
abt01_test = pd.read_parquet(PROCESSED_DIR / 'abt01_test_fs.parquet')

print(f'✅ Dados de Treino: {abt01_train.shape[0]:,} registros × {abt01_train.shape[1]} features')
print(f'\nDistribuição do Target (FPD):')
print(f"  Treino: {(abt01_train['FPD'].value_counts(normalize=True) * 100).round(2)}")

## Verificando força das variáveis explicativas com IV

In [ ]:
# Exemplo de uso:
iv_df = iv_table(abt01_train, TARGET)
iv_df

## Linearidade com Log(Odds)

In [ ]:
# Calcula o R² do ajuste log-odds e filtra variáveis com estabilidade >= threshold
r2_df = calculate_r2_for_logodds(abt01_train, list(iv_df.Variável), TARGET, threshold=0.85)
r2_df

In [ ]:
# Lista variáveis que precisam ser categorizadas com base no critério de R² do log-odds
categorize_vars = r2_df[r2_df['Feat Eng'] == 'Categorizar']['Variable'].tolist()
print(categorize_vars)

In [ ]:
# Define estratégia de feature engineering com base na qualidade do ajuste log-odds
results_df = calculate_r2_for_logodds_and_transformations(abt01_train, categorize_vars, TARGET, threshold=0.8)

In [ ]:
# Extrai variáveis contínuas aprovadas e a transformação ótima para aplicar no pipeline
transform_map = (results_df.query("`Feat Eng` == 'Usar como contínua'").set_index('Variable')['Best Transformation'].to_dict())

In [ ]:
# Aplica transformações no treino
abt01_train_transformed = apply_transformations_from_map(abt01_train, transform_map, drop_original=True)

In [ ]:
# Lista variáveis que precisam ser categorizadas com base no critério de R² do log-odds
categorize_vars = results_df[results_df['Feat Eng'] == 'Categorizar']['Variable'].tolist()

# Remove variáveis marcadas para categorização do dataset base
abt01_train_transformed = abt01_train_transformed.drop(columns=categorize_vars)

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'feature_transformations.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(transform_map, f)

In [ ]:
# salva a lista final de features usadas pelo modelo (exclui a target)
artifact_path = Path(ARTIFACT_DIR) / 'model_features.pkl'

model_features = [c for c in abt01_train_transformed.columns if c != TARGET]
with open(artifact_path, 'wb') as f:
    pickle.dump(model_features, f)

print(f'✓ Lista de features do modelo salva em: {artifact_path}')
print('✅ Pipeline de preparação de dados finalizado')

In [ ]:
# carrega transformações salvas e aplica no dataset de teste
with open(Path(ARTIFACT_DIR) / 'feature_transformations.pkl', 'rb') as f:
    transform_map_load = pickle.load(f)

# Aplica transformações no teste
abt01_test_transformed = apply_transformations_from_map(abt01_test, transform_map_load, drop_original=True)

# Remove variáveis marcadas para categorização do dataset base
abt01_test_transformed = abt01_test_transformed.drop(columns=categorize_vars)

## Salvamento dos Dados Processados

In [ ]:
# Salvar datasets processados e lista de features

print('\n💾 Salvando dados processados...')

# Dataset completo (opcional)
abt01_train_transformed.to_parquet(PROCESSED_DIR / 'abt01_train_tr.parquet', index=False)
abt01_test_transformed.to_parquet(PROCESSED_DIR / 'abt01_test_tr.parquet', index=False)
print(f'   ✓ Treino salvo: {PROCESSED_DIR / "abt01_train_tr.parquet"}')
print(f'   ✓ Teste salvo: {PROCESSED_DIR / "abt01_test_tr.parquet"}')

# salva a lista final de features usadas pelo modelo (exclui a target)
artifact_path = Path(ARTIFACT_DIR) / 'model_features.pkl'

model_features = [c for c in abt01_train_transformed.columns if c != TARGET]
with open(artifact_path, 'wb') as f:
    pickle.dump(model_features, f)

print(f'✓ Lista de features do modelo salva em: {artifact_path}')
print('✅ Pipeline de preparação de dados finalizado')